# Time-Series Embedding — Contrastive Training Starter

End-to-end walkthrough using the `ts_embed` package:

1. Generate **small synthetic data** in the exact memmap format the dataset expects
   (real data: 3M accounts × 24 months × 100 features — here we use a tiny sample).
2. Build the dataset, contrastive masking collator, model, and VICReg loss.
3. Run a short training loop on a single GPU (falls back to CPU for the demo).
4. Extract embeddings and **sanity-check for representation collapse**.

Swap the synthetic arrays in Step 1 for your real `numeric.npy / missing.npy /
categorical.npy` and scale up the config to use this as a real starting point.

In [ ]:
import sys, pathlib
# Make the repo root importable when running the notebook from anywhere.
REPO_ROOT = pathlib.Path.cwd()
if not (REPO_ROOT / 'ts_embed').exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import torch
from torch.utils.data import DataLoader

from ts_embed.data import DatasetPaths, TimeSeriesDataset, TimeFeatureMasker, ContrastiveCollator
from ts_embed.model import TSEmbeddingModel, TSEncoderConfig
from ts_embed.loss import VICRegLoss, VICRegConfig

torch.manual_seed(0)
np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## 1. Synthetic data in the expected on-disk format

The dataset reads three memmapped `.npy` files. We fabricate a small set with
latent structure (two clusters) so we can later verify the embeddings separate
them — a quick proxy for “the model learned something, not a constant.”

In [ ]:
N, T, F_NUM, F_CAT = 4000, 24, 98, 2     # demo scale; real: N=3_000_000
MISSING_RATE = 0.1

data_dir = REPO_ROOT / 'data_demo'
data_dir.mkdir(exist_ok=True)

# Two latent clusters with different temporal trends.
group = np.random.randint(0, 2, size=N)
t_axis = np.linspace(0, 1, T)[None, :, None]
trend = np.where(group[:, None, None] == 0, t_axis, -t_axis)
numeric = (trend + 0.3 * np.random.randn(N, T, F_NUM)).astype(np.float32)

# Original missingness, then 'impute' with the per-feature mean.
missing = (np.random.rand(N, T, F_NUM) < MISSING_RATE).astype(np.uint8)
feat_mean = numeric.mean(axis=(0, 1), keepdims=True)
numeric = np.where(missing == 1, feat_mean, numeric).astype(np.float32)

# Binary categoricals correlated with the latent group.
categorical = np.stack([
    (np.random.rand(N, T) < (0.2 + 0.6 * group[:, None])).astype(np.int8),
    (np.random.rand(N, T) < 0.5).astype(np.int8),
], axis=-1)

np.save(data_dir / 'numeric.npy', numeric)
np.save(data_dir / 'missing.npy', missing)
np.save(data_dir / 'categorical.npy', categorical)
print({p.name: np.load(p, mmap_mode='r').shape for p in sorted(data_dir.glob('*.npy'))})

## 2. Dataset + contrastive collator + model + loss

`ContrastiveCollator` produces two views per account: view A (no masking) and
view B (time + feature masking). The masked cells reuse the missing-data
pathway. VICReg's variance/covariance terms prevent collapse.

In [ ]:
paths = DatasetPaths(
    numeric=data_dir / 'numeric.npy',
    missing=data_dir / 'missing.npy',
    categorical=data_dir / 'categorical.npy',
)
dataset = TimeSeriesDataset(paths)

masker = TimeFeatureMasker(time_mask_prob=0.25, feature_mask_prob=0.30)
collator = ContrastiveCollator(masker)

loader = DataLoader(
    dataset, batch_size=256, shuffle=True, drop_last=True,
    num_workers=0,          # 0 for notebook; bump to 8+ in scripts
    collate_fn=collator,
)

# Smaller model for the demo; scale d_model/n_layers up for real data.
cfg = TSEncoderConfig(n_numeric=F_NUM, cat_cardinalities=(2, 2), seq_len=T,
                      d_model=96, n_layers=2, n_heads=4, proj_dim=128)
model = TSEmbeddingModel(cfg).to(device)
loss_fn = VICRegLoss(VICRegConfig(gather_distributed=False))
optim = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

n_params = sum(p.numel() for p in model.parameters())
print(f'model params: {n_params/1e6:.2f}M | steps/epoch: {len(loader)}')

## 3. Training loop

Watch the three loss components:
- **sim** ↓ — masked and unmasked views converge.
- **var** should stay low (near 0) — if it rises and stays high the model is
  collapsing; raise `var_coef`.
- **cov** ↓ — embedding dimensions decorrelate.

In [ ]:
EPOCHS = 8
history = []
model.train()
for epoch in range(EPOCHS):
    for batch in loader:
        num_a = batch['numeric_a'].to(device); mis_a = batch['missing_a'].to(device)
        keep_a = batch['time_keep_a'].to(device)
        num_b = batch['numeric_b'].to(device); mis_b = batch['missing_b'].to(device)
        keep_b = batch['time_keep_b'].to(device)
        cat_a = batch['categorical_a'].to(device)
        cat_b = batch['categorical_b'].to(device)

        optim.zero_grad(set_to_none=True)
        _, z_a = model(num_a, mis_a, cat_a, keep_a)
        _, z_b = model(num_b, mis_b, cat_b, keep_b)
        losses = loss_fn(z_a, z_b)
        losses['loss'].backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optim.step()

    history.append({k: float(v) for k, v in losses.items()})
    print(f"epoch {epoch}: loss={losses['loss']:.3f} sim={losses['sim']:.3f} "
          f"var={losses['var']:.3f} cov={losses['cov']:.3f}")

In [ ]:
# Optional: plot the loss curves.
try:
    import matplotlib.pyplot as plt
    for k in ['loss', 'sim', 'var', 'cov']:
        plt.plot([h[k] for h in history], label=k)
    plt.legend(); plt.xlabel('epoch'); plt.title('VICReg components'); plt.show()
except ImportError:
    print('matplotlib not installed; skipping plot')

## 4. Extract embeddings + collapse sanity-check

Use the **encoder output** (not the projector) as the deliverable embedding.
Two checks:
1. Per-dimension std ≫ 0 — a near-zero std means collapse.
2. The two synthetic latent groups should be separable in embedding space.

In [ ]:
def collate_eval(b):
    return {
        'numeric': torch.stack([x['numeric'] for x in b]),
        'missing': torch.stack([x['missing'] for x in b]),
        'categorical': torch.stack([x['categorical'] for x in b]),
    }

eval_loader = DataLoader(dataset, batch_size=512, shuffle=False,
                         num_workers=0, collate_fn=collate_eval)
model.eval()
embs = []
with torch.inference_mode():
    for b in eval_loader:
        z = model.encode(b['numeric'].to(device), b['missing'].to(device),
                          b['categorical'].to(device))
        embs.append(z.float().cpu().numpy())
embs = np.concatenate(embs)
print('embeddings:', embs.shape)

per_dim_std = embs.std(axis=0)
print(f'per-dim std  mean={per_dim_std.mean():.4f}  min={per_dim_std.min():.4f}')
assert per_dim_std.mean() > 1e-2, 'COLLAPSE: embeddings nearly constant — raise var_coef'

# Group separability: distance between class means vs within-class spread.
m0, m1 = embs[group == 0].mean(0), embs[group == 1].mean(0)
between = np.linalg.norm(m0 - m1)
within = np.linalg.norm(embs - np.where(group[:, None] == 0, m0, m1), axis=1).mean()
print(f'between-group dist={between:.3f}  within-group dist={within:.3f}  '
      f'ratio={between/within:.2f} (higher = more structure learned)')

## Next steps for the real project

- **Data**: replace Step 1 with your real `numeric.npy / missing.npy /
  categorical.npy` (build them once during data prep; keep memmap layout).
- **Scale config**: `d_model=192, n_layers=4, n_heads=6` is a sane starting
  point at 3M accounts; tune from there.
- **Single GPU**: `python scripts/train_single_gpu.py --numeric ... --out runs/sg`
- **Cluster**: `torchrun --nnodes=N --nproc_per_node=8 scripts/train_ddp.py ...`
  (sharded sampler, SyncBatchNorm, cross-rank VICReg stats).
- **Production embeddings**: `python scripts/extract_embeddings.py --ckpt ...`
- **If collapse appears** (var loss high, per-dim std → 0): raise `var_coef`
  in `VICRegConfig`, lower LR, or reduce mask probabilities.